In [ ]:
import os
import crewai
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM
from crewai_tools import SerperDevTool, ScrapeWebsiteTool

# Load your hidden .env file for the search API key
load_dotenv()
SERPER_API_KEY = os.getenv("SERPER_API_KEY")
os.environ["SERPER_API_KEY"] = SERPER_API_KEY

# Initialize tools required for Phase 1 Desirability market analysis
search_tool = SerperDevTool(api_key=SERPER_API_KEY)
scrape_tool = ScrapeWebsiteTool()

In [ ]:
# 1. Structured Project Requirement Input (Sample data based on system specifications)
STUDENT_INPUT = {
    "project_title": "AI Entrepreneurship Coach",
    "problem_statement": "Early-stage student founders lack affordable, instant, and structured guidance to validate their business ideas. Human mentorship is highly bottlenecked, expensive, and unavailable at flexible hours, leading to premature failure.",
    "target_users": "University students, first-time startup founders, and college incubator E-Cells.",
    "expected_features": "24/7 conversational chat interface, automated interactive Business Model Canvas generator, automated SWOT analysis creator, and a pitch script evaluator backed by VC standards.",
    "business_objective": "Provide a low-cost, automated decision-support pipeline to filter weak ideas and strengthen viable ones before human mentorship intervention."
}

# 2. Configure CrewAI to point to your local LM Studio server
# Critical Note: Switch your local Bonsai-8B model to at least Q4_K_M quantization if you experience reasoning loops.
llm = LLM(
    model="bonsai-8B", 
    base_url="http://127.0.0.1:1234/v1", 
    api_key="lm-studio"                  
)

# 3. Define the Phase 1 Desirability Evaluation Agent
desirability_agent = Agent(
    role="Desirability Evaluation Agent",
    goal="Determine whether the proposed solution addresses a genuine user need and whether sufficient market demand exists.",
    backstory=(
        "You are an expert market research analyst and user experience strategist. "
        "Your job is to critically analyze early-stage ideas, identify real user pain points, "
        "assess market demand trends, and research existing competitor alternatives to find product-market fit gaps."
    ),
    llm=llm,
    tools=[search_tool, scrape_tool],
    verbose=True,
    max_iter=4  # Optimized to allow deep research without freezing the local machine
)

# 4. Define the Desirability Task mapped exactly to system documentation outputs
desirability_task = Task(
    description=(
        f"Analyze the following student project proposal:\n"
        f"- Title: {STUDENT_INPUT['project_title']}\n"
        f"- Problem Statement: {STUDENT_INPUT['problem_statement']}\n"
        f"- Target Users: {STUDENT_INPUT['target_users']}\n"
        f"- Features: {STUDENT_INPUT['expected_features']}\n"
        f"- Objective: {STUDENT_INPUT['business_objective']}\n\n"
        "Execution Requirements:\n"
        "1. Use the search tool to find 2-3 existing competitor platforms, apps, or manual alternatives addressing this problem.\n"
        "2. Scrape or analyze current market trend resources to verify if search interest or industry demand for automated coaching/AI tools is growing.\n"
        "3. Evaluate the severity of the target user pain points and assess if the features directly solve them.\n"
        "4. Synthesize all findings into a clear, cohesive report."
    ),
    expected_output=(
        "A formal text-formatted 'Desirability Analysis Report' containing:\n"
        "1. User Demand Analysis (validating target user pain points and problem severity).\n"
        "2. Market Demand Assessment (current industry search interest and growth indicators).\n"
        "3. Competitor Analysis (gaps, weaknesses, or friction in existing products/alternatives).\n"
        "4. Opportunity Identification (clear statement on why this solution is or is not desired by the market)."
    ),
    agent=desirability_agent
)


In [ ]:
feasibility_agent = Agent(
        role="Feasibility Evaluation Agent",
        goal="Evaluate the feasibility of a startup idea strictly from the Feasibility dimension of the DFV framework.",
        backstory=(
            "You are an expert technical architect, systems analyst, and execution strategist. "
            "Your task is to assess whether a startup idea can realistically be built and operated. "
            "You only evaluate the Feasibility part of DFV. "
            "Do not evaluate desirability or viability. "
            "Do not give ratings, scores, grades, or percentages. "
            "If the idea has technical or operational weaknesses, clearly explain them and suggest improvements."
        ),
        llm=llm,
        verbose=True,
        max_iter=4
    )

feasibility_task = Task(
        description=(
            f"The following is a startup/project idea submitted by a user:\n\n"
            f"'{user_idea}'\n\n"
            "Your job is to evaluate this idea strictly from the FEASIBILITY perspective of the DFV framework.\n\n"
            "Focus only on:\n"
            "1. Whether the product can realistically be built with current technology.\n"
            "2. What tech stack, tools, models, APIs, or infrastructure may be required.\n"
            "3. What technical and operational challenges may arise.\n"
            "4. Whether the scope is too broad, unrealistic, or difficult for a student/startup team.\n"
            "5. What changes or simplifications would make the idea more feasible.\n\n"
            "Important constraints:\n"
            "- Do NOT evaluate desirability.\n"
            "- Do NOT evaluate viability.\n"
            "- Do NOT give a score, rating, grade, percentage, or rank.\n"
            "- If the idea is weak, provide constructive suggestions.\n"
            "- If the idea is feasible, explain why and suggest next implementation steps."
        ),
        expected_output=(
            "A plain-language Feasibility Evaluation containing:\n"
            "1. A short feasibility opinion.\n"
            "2. Main technical and operational challenges.\n"
            "3. Required tools, stack, or infrastructure.\n"
            "4. Suggestions to improve or simplify the idea if needed.\n"
            "5. Practical next steps for implementation.\n"
            "Do not include any score, rating, grade, or percentage."
        ),
        agent=feasibility_agent
    )

In [ ]:
viability_agent = Agent(
    role="Viability Evaluation Agent",
    goal="Determine whether the proposed solution can generate sustainable business value and long-term growth.",
    backstory=(
        "You are an expert startup strategist, business consultant, and commercialization analyst. "
        "Your responsibility is to evaluate business models, revenue opportunities, customer segments, "
        "cost considerations, market sustainability, and long-term commercial success."
    ),
    llm=llm,
    tools=[search_tool, scrape_tool],
    verbose=True,
    max_iter=4
)

viability_task = Task(
    description=(
        f"Analyze the business viability of the following project proposal:\n"
        f"- Title: {STUDENT_INPUT['project_title']}\n"
        f"- Problem Statement: {STUDENT_INPUT['problem_statement']}\n"
        f"- Target Users: {STUDENT_INPUT['target_users']}\n"
        f"- Features: {STUDENT_INPUT['expected_features']}\n"
        f"- Objective: {STUDENT_INPUT['business_objective']}\n\n"

        "Execution Requirements:\n"
        "1. Identify potential paying customer segments.\n"
        "2. Identify suitable business models.\n"
        "3. Analyze possible revenue streams.\n"
        "4. Assess market size and growth potential.\n"
        "5. Evaluate cost considerations.\n"
        "6. Evaluate long-term sustainability.\n"
        "7. Provide a final viability conclusion."
    ),

    expected_output=(
        "A Viability Analysis Report containing:\n"
        "1. Business Model Analysis\n"
        "2. Revenue Opportunities\n"
        "3. Customer Segment Analysis\n"
        "4. Cost Considerations\n"
        "5. Sustainability Assessment\n"
        "6. Final Viability Conclusion"
    ),

    agent=viability_agent
)

In [ ]:
dfv_risk_decision_agent = Agent(
    role="Internal DFV Decision and Risk Assessment Engine",
    goal="Identify hidden risks across project dimensions and aggregate all findings into a final project readiness decision.",
    backstory=(
        "You are an expert venture risk analyst and product strategist. Your superpower is looking "
        "at a product's user demand, technical stack, and business model, and instantly identifying "
        "where things could go wrong (e.g., API limits, low adoption, or high maintenance costs). "
        "You take a supportive, coaching approach: if you find critical risks, you don't just stop the project—"
        "you provide a highly positive, encouraging roadmap on how to mitigate those risks and improve the idea."
    ),
    verbose=True,
    llm=llm
)

dfv_decision_task = Task(
    description=(
        "Review the reports provided in your context thoroughly from the Desirability, "
        "Feasibility, and Viability evaluation phases.\n\n"
        
        "STEP 1: Perform a Risk Assessment based on those reports. Identify potential:\n"
        "- Technical Risks (e.g., API constraints, hallucinations, scalability issues)\n"
        "- Business Risks (e.g., market competition, adoption barriers)\n"
        "- Operational Risks (e.g., infrastructure or maintenance overhead costs)\n\n"
        
        "STEP 2: Synthesize the risks with the core DFV dimensions and determine if the "
        "overall project is a 'GO' (Proceed) or 'NO-GO' (Needs Improvement)."
    ),
    expected_output=(
        "A structured markdown report using the following exact format:\n\n"
        "## Final Decision: [GO / NO-GO]\n\n"
        "### Executive Summary\n"
        "[A concise evaluation summary of the overall project strength and readiness.]\n\n"
        
        "### Internal Risk Assessment Summary\n"
        "* **Technical Risks Identified:** [Brief takeaway or 'None identified']\n"
        "* **Business/Operational Risks Identified:** [Brief takeaway or 'None identified']\n\n"
        
        "### Dimension Breakdown\n"
        "* **Desirability Summary:** [Brief takeaway from the context report]\n"
        "* **Feasibility Summary:** [Brief takeaway from the context report]\n"
        "* **Viability Summary:** [Brief takeaway from the context report]\n\n"
        
        "### Actionable Recommendations (Only if NO-GO)\n"
        "[Provide a highly positive, encouraging, and constructive bulleted list of changes, "
        "pivots, or mitigation strategies the team can implement to fix the gaps and safely proceed. "
        "If the decision is GO, state 'No major adjustments needed; proceed with identified mitigations.']\n\n"
        "**CRITICAL RULE:** Do not include or expose any internal numerical scoring or percentages."
    ),
    # This is how data is passed between tasks in CrewAI:
    context=[desirability_task, feasibility_task, viability_task],
    agent=dfv_risk_decision_agent
)

In [ ]:
# 5. Assemble the Phase 1 Crew
crew = Crew(
    agents=[desirability_agent, feasibility_agent, viability_agent, dfv_risk_decision_agent],
    tasks=[desirability_task, feasibility_task, viability_task, dfv_decision_task],
    process=Process.sequential,
    verbose=True
)


# Run the Crew using top-level notebook await structure
print(f"--- STARTING DESIRABILITY EVALUATION FOR: {STUDENT_INPUT['project_title']} ---\n")
result = await crew.kickoff_async()

print("\n--- FINAL DESIRABILITY ANALYSIS REPORT ---\n")
print(result)